In [ ]:
import os, numpy as np, pandas as pd, tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dropout, GlobalAveragePooling2D, Dense, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, CSVLogger
from tensorflow.keras.optimizers import Adam
from google.colab import drive
drive.mount('/content/drive')
print("✅ Model2 Setup Ready")

Mounted at /content/drive
✅ Model2 Setup Ready


In [ ]:
base_dir = '/content/drive/MyDrive/Robust_MHCNNFD_Dataset'
train_dir, val_dir, test_dir = [os.path.join(base_dir, s) for s in ['train', 'val', 'test']]

print("📊 Model2 Dataset Audit (25,935 images)")
for split, path in zip(['TRAIN','VAL','TEST'], [train_dir,val_dir,test_dir]):
    print(f"\n📂 {split}: {path}")
    if os.path.exists(path):
        total = sum(len(os.listdir(os.path.join(path,c))) for c in os.listdir(path) if os.path.isdir(os.path.join(path,c)))
        print(f"   Total: {total}")
print("🟢 Verified ✓")

📊 Model2 Dataset Audit (25,935 images)

📂 TRAIN: /content/drive/MyDrive/Robust_MHCNNFD_Dataset/train
   Total: 20745

📂 VAL: /content/drive/MyDrive/Robust_MHCNNFD_Dataset/val
   Total: 2595

📂 TEST: /content/drive/MyDrive/Robust_MHCNNFD_Dataset/test
   Total: 2595
🟢 Verified ✓


In [ ]:
datagen = ImageDataGenerator(rescale=1./255)
BATCH_SIZE, IMG_SIZE = 32, (256, 256)

train_gen = datagen.flow_from_directory(train_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
                                        class_mode='categorical', shuffle=True)
val_gen = datagen.flow_from_directory(val_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
                                      class_mode='categorical', shuffle=False)
test_gen = datagen.flow_from_directory(test_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
                                       class_mode='categorical', shuffle=False)

print(f"✅ Generators: Train={train_gen.samples} Val={val_gen.samples} Test={test_gen.samples}")
print("Classes:", train_gen.class_indices)

Found 20745 images belonging to 4 classes.
Found 2595 images belonging to 4 classes.
Found 2595 images belonging to 4 classes.
✅ Generators: Train=20745 Val=2595 Test=2595
Classes: {'Evening_Fire_Incident_aug_img': 0, 'Evening_Forest_Condition_aug_img': 1, 'Pre-_Evening_Fire_Incident_aug_img': 2, 'Pre-_Evening_Forest_Condition_aug_img': 3}


In [ ]:
def build_model2():
    inputs = Input((256,256,3))
    x = Conv2D(64,(3,3),padding='valid',activation='relu')(inputs); x=BatchNormalization()(x); x=MaxPooling2D(2,2)(x)
    x = Conv2D(128,(3,3),padding='valid',activation='relu')(x); x=BatchNormalization()(x); x=MaxPooling2D(2,2)(x)
    x = Conv2D(256,(3,3),padding='valid',activation='relu')(x); x=BatchNormalization()(x); x=MaxPooling2D(2,2)(x)
    x = Conv2D(128,(3,3),padding='valid',activation='relu')(x); x=BatchNormalization()(x)
    x = Conv2D(64,(3,3),padding='valid',activation='relu')(x); x=BatchNormalization()(x); x=Dropout(0.3)(x)
    x = GlobalAveragePooling2D()(x)
    x = Dense(64,activation='selu')(x); x=Dropout(0.3)(x)
    outputs = Dense(4,activation='softmax')(x)
    return Model(inputs,outputs,name="Model2_4Class")

model2 = build_model2()
model2.compile(optimizer=Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])
model2.summary()
print("✅ Model2 Ready (752K params)")

Model: "Model2_4Class"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 254, 254, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 254, 254, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 127, 127, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 125, 125, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 125, 125, 128)  │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 62, 62, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 60, 60, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 60, 60, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 30, 30, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 28, 28, 128)    │       295,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 28, 28, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 26, 26, 64)     │        73,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 26, 26, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 26, 26, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 746,628 (2.85 MB)

 Trainable params: 745,348 (2.84 MB)

 Non-trainable params: 1,280 (5.00 KB)

✅ Model2 Ready (752K params)


In [ ]:
save_folder = '/content/drive/MyDrive/Model2_Robust_Training'
os.makedirs(save_folder, exist_ok=True)

checkpoint = ModelCheckpoint(os.path.join(save_folder, 'Model2_Epoch_{epoch:02d}_ValAcc_{val_accuracy:.4f}.keras'),
                             monitor='val_accuracy', save_best_only=False, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6, verbose=1)
csv_logger = CSVLogger(os.path.join(save_folder, 'Model2_History.csv'), append=True)

history = model2.fit(train_gen, validation_data=val_gen, epochs=40,
                     callbacks=[checkpoint, reduce_lr, csv_logger])
print("🎉 Training Done!")

Epoch 1/40
649/649 ━━━━━━━━━━━━━━━━━━━━ 0s 22s/step - accuracy: 0.7899 - loss: 0.5141 
Epoch 1: saving model to /content/drive/MyDrive/Model2_Robust_Training/Model2_Epoch_01_ValAcc_0.7118.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Model2_Robust_Training/Model2_Epoch_01_ValAcc_0.7118.keras
649/649 ━━━━━━━━━━━━━━━━━━━━ 15323s 24s/step - accuracy: 0.8694 - loss: 0.3383 - val_accuracy: 0.7118 - val_loss: 1.0986 - learning_rate: 0.0010
Epoch 2/40
649/649 ━━━━━━━━━━━━━━━━━━━━ 0s 22s/step - accuracy: 0.9486 - loss: 0.1515 
Epoch 2: saving model to /content/drive/MyDrive/Model2_Robust_Training/Model2_Epoch_02_ValAcc_0.5592.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Model2_Robust_Training/Model2_Epoch_02_ValAcc_0.5592.keras
649/649 ━━━━━━━━━━━━━━━━━━━━ 14848s 23s/step - accuracy: 0.9566 - loss: 0.1293 - val_accuracy: 0.5592 - val_loss: 4.1826 - learning_rate: 0.0010
Epoch 3/40
591/649 ━━━━━━━━━━━━━━━━━━━━ 21:16 22s/step - accuracy: 0.9572 - loss: 0.1263

In [ ]:
#resume

In [ ]:
save_folder = '/content/drive/MyDrive/Model2_Robust_Training'
os.makedirs(save_folder, exist_ok=True)

checkpoint = ModelCheckpoint(os.path.join(save_folder, 'Model2_Epoch_{epoch:02d}_ValAcc_{val_accuracy:.4f}.keras'),
                             monitor='val_accuracy', save_best_only=False, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6, verbose=1)
csv_logger = CSVLogger(os.path.join(save_folder, 'Model2_History.csv'), append=True)

history = model2.fit(train_gen, validation_data=val_gen, epochs=40,
                     initial_epoch=2,  # ← Starts Epoch 3
                     callbacks=[checkpoint, reduce_lr, csv_logger])
print("🎉 Model2 Training Complete!")

Epoch 3/40
649/649 ━━━━━━━━━━━━━━━━━━━━ 0s 22s/step - accuracy: 0.7962 - loss: 0.5091 
Epoch 3: saving model to /content/drive/MyDrive/Model2_Robust_Training/Model2_Epoch_03_ValAcc_0.5580.keras

Epoch 3: finished saving model to /content/drive/MyDrive/Model2_Robust_Training/Model2_Epoch_03_ValAcc_0.5580.keras
649/649 ━━━━━━━━━━━━━━━━━━━━ 15066s 23s/step - accuracy: 0.8816 - loss: 0.3179 - val_accuracy: 0.5580 - val_loss: 2.0648 - learning_rate: 0.0010
Epoch 4/40
649/649 ━━━━━━━━━━━━━━━━━━━━ 0s 22s/step - accuracy: 0.9493 - loss: 0.1465 
Epoch 4: saving model to /content/drive/MyDrive/Model2_Robust_Training/Model2_Epoch_04_ValAcc_0.9098.keras

Epoch 4: finished saving model to /content/drive/MyDrive/Model2_Robust_Training/Model2_Epoch_04_ValAcc_0.9098.keras
649/649 ━━━━━━━━━━━━━━━━━━━━ 14471s 22s/step - accuracy: 0.9566 - loss: 0.1268 - val_accuracy: 0.9098 - val_loss: 0.5474 - learning_rate: 0.0010
Epoch 5/40
622/649 ━━━━━━━━━━━━━━━━━━━━ 9:47 22s/step - accuracy: 0.9645 - loss: 0.1065 

In [ ]:
save_folder = '/content/drive/MyDrive/Model2_Robust_Training'
os.makedirs(save_folder, exist_ok=True)

checkpoint = ModelCheckpoint(os.path.join(save_folder, 'Model2_Epoch_{epoch:02d}_ValAcc_{val_accuracy:.4f}.keras'),
                             monitor='val_accuracy', save_best_only=False, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6, verbose=1)
csv_logger = CSVLogger(os.path.join(save_folder, 'Model2_History.csv'), append=True)

history = model2.fit(train_gen, validation_data=val_gen, epochs=40,
                     initial_epoch=4,  # ← Starts Epoch 5
                     callbacks=[checkpoint, reduce_lr, csv_logger])
print("🎉 Model2 Training Complete!")

Epoch 5/40
649/649 ━━━━━━━━━━━━━━━━━━━━ 0s 28s/step - accuracy: 0.8115 - loss: 0.4860 
Epoch 5: saving model to /content/drive/MyDrive/Model2_Robust_Training/Model2_Epoch_05_ValAcc_0.8794.keras

Epoch 5: finished saving model to /content/drive/MyDrive/Model2_Robust_Training/Model2_Epoch_05_ValAcc_0.8794.keras
649/649 ━━━━━━━━━━━━━━━━━━━━ 20741s 32s/step - accuracy: 0.8852 - loss: 0.3143 - val_accuracy: 0.8794 - val_loss: 0.3377 - learning_rate: 0.0010
Epoch 6/40
649/649 ━━━━━━━━━━━━━━━━━━━━ 0s 18s/step - accuracy: 0.9522 - loss: 0.1359 
Epoch 6: saving model to /content/drive/MyDrive/Model2_Robust_Training/Model2_Epoch_06_ValAcc_0.6324.keras

Epoch 6: finished saving model to /content/drive/MyDrive/Model2_Robust_Training/Model2_Epoch_06_ValAcc_0.6324.keras
649/649 ━━━━━━━━━━━━━━━━━━━━ 11898s 18s/step - accuracy: 0.9532 - loss: 0.1350 - val_accuracy: 0.6324 - val_loss: 3.4756 - learning_rate: 0.0010
Epoch 7/40
572/649 ━━━━━━━━━━━━━━━━━━━━ 22:50 18s/step - accuracy: 0.9675 - loss: 0.0937

In [ ]:
save_folder = '/content/drive/MyDrive/Model2_Robust_Training'
os.makedirs(save_folder, exist_ok=True)

checkpoint = ModelCheckpoint(os.path.join(save_folder, 'Model2_Epoch_{epoch:02d}_ValAcc_{val_accuracy:.4f}.keras'),
                             monitor='val_accuracy', save_best_only=False, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6, verbose=1)
csv_logger = CSVLogger(os.path.join(save_folder, 'Model2_History.csv'), append=True)

history = model2.fit(train_gen, validation_data=val_gen, epochs=40,
                     initial_epoch=6,  # ← Starts Epoch 7
                     callbacks=[checkpoint, reduce_lr, csv_logger])
print("🎉 Model2 Training Complete!")

Epoch 7/40
649/649 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.8115 - loss: 0.4716
Epoch 7: saving model to /content/drive/MyDrive/Model2_Robust_Training/Model2_Epoch_07_ValAcc_0.6659.keras

Epoch 7: finished saving model to /content/drive/MyDrive/Model2_Robust_Training/Model2_Epoch_07_ValAcc_0.6659.keras
649/649 ━━━━━━━━━━━━━━━━━━━━ 6011s 9s/step - accuracy: 0.8883 - loss: 0.2970 - val_accuracy: 0.6659 - val_loss: 2.1437 - learning_rate: 0.0010
Epoch 8/40
649/649 ━━━━━━━━━━━━━━━━━━━━ 0s 583ms/step - accuracy: 0.9512 - loss: 0.1543
Epoch 8: saving model to /content/drive/MyDrive/Model2_Robust_Training/Model2_Epoch_08_ValAcc_0.4439.keras

Epoch 8: finished saving model to /content/drive/MyDrive/Model2_Robust_Training/Model2_Epoch_08_ValAcc_0.4439.keras
649/649 ━━━━━━━━━━━━━━━━━━━━ 424s 653ms/step - accuracy: 0.9574 - loss: 0.1287 - val_accuracy: 0.4439 - val_loss: 12.2261 - learning_rate: 0.0010
Epoch 9/40
 97/649 ━━━━━━━━━━━━━━━━━━━━ 5:44 625ms/step - accuracy: 0.9620 - loss: 0.1282

In [ ]:
save_folder = '/content/drive/MyDrive/Model2_Robust_Training'
os.makedirs(save_folder, exist_ok=True)

checkpoint = ModelCheckpoint(os.path.join(save_folder, 'Model2_Epoch_{epoch:02d}_ValAcc_{val_accuracy:.4f}.keras'),
                             monitor='val_accuracy', save_best_only=False, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6, verbose=1)
csv_logger = CSVLogger(os.path.join(save_folder, 'Model2_History.csv'), append=True)

history = model2.fit(train_gen, validation_data=val_gen, epochs=40,
                     initial_epoch=8,  # ← Starts Epoch 9
                     callbacks=[checkpoint, reduce_lr, csv_logger])
print("🎉 Model2 Training Complete!")

Epoch 9/40
649/649 ━━━━━━━━━━━━━━━━━━━━ 0s 18s/step - accuracy: 0.8020 - loss: 0.5014 
Epoch 9: saving model to /content/drive/MyDrive/Model2_Robust_Training/Model2_Epoch_09_ValAcc_0.8497.keras

Epoch 9: finished saving model to /content/drive/MyDrive/Model2_Robust_Training/Model2_Epoch_09_ValAcc_0.8497.keras
649/649 ━━━━━━━━━━━━━━━━━━━━ 12620s 19s/step - accuracy: 0.8813 - loss: 0.3181 - val_accuracy: 0.8497 - val_loss: 0.4466 - learning_rate: 0.0010
Epoch 10/40
494/649 ━━━━━━━━━━━━━━━━━━━━ 46:48 18s/step - accuracy: 0.9523 - loss: 0.1398